#### Import General Functions

In [ ]:
from scipy.integrate import solve_ivp
import numpy as np
import pandas as pd

#### Import Functions and Dictionaries

In [ ]:
from ODESystem_Final import SnyderODE

#### Function

In [1]:
def ODEfunction(
    NUMYEARS,
    time_points,  
    init_cond,
#    dictionary,
#    dictionary,
    dictionary,
    function):


    outputs = []

    #Benthos Initial Conditions-----------------------------------------------------
    C_init = init_cond['C_init']
    C0, M0, T0 = C_init, ((1-C_init)/2), ((1-C_init)/2) 

    
    #Herbivore Initial Conditions-----------------------------------------------------
        
    for B in range(101):
        
        for G in range(101 - B):
            
            S = 100 - B - G
            
            if S >= 0:
                B0 = B / 100
                G0 = G / 100
                S0 = S / 100
    
            
              #Benthos Parameters-----------------------------------------------------
                rC  = dictionary['rC']
                αCT = dictionary['αCT']
                μC  = dictionary['μC']
                rM  = dictionary['rM']
                αMT = dictionary['αMT']
                αMC = dictionary['αMC']
                rT  = dictionary['rT']

                
                #Herbivore Parameters-----------------------------------------------------
                #Browser
                sB = dictionary['sB']
                hB = dictionary['hB']
                ηB = dictionary['ηB']
                eB = dictionary['eB']
                σB = dictionary['σB']
                θB = dictionary['θB']
                γB = dictionary['γB']
                μB = dictionary['μB']

                #Grazer
                sG = dictionary['sG']
                hG = dictionary['hG']
                ηG = dictionary['ηG']
                eG = dictionary['eG']
                σG = dictionary['σG']
                θG = dictionary['θG']
                γG = dictionary['γG']
                μG = dictionary['μG']

                #Scraper
                sS = dictionary['sS']
                hS = dictionary['hS']
                ηS = dictionary['ηS']
                eS = dictionary['eS']
                σS = dictionary['σS']
                θS = dictionary['θS']
                γS = dictionary['γS']
                μS = dictionary['μS']

                                
                #Solve ODE-----------------------------------------------------
                #here, you turned off dense_output (and changed N = sol.sol(timepoints) to N = sol.y) 
                #and defined t_eval so that you can get rates of change at each timestep 
                #i.e., if dense_output = true, it does adaptive step-sizing
                
                sol = solve_ivp(function, [0,NUMYEARS], [C0, M0, T0, B0, G0, S0], 
                                                t_eval = time_points,
                                                method = 'RK45', 
                                                args = (rC, rM, rT, αCT, αMT, αMC, μC,              #benthos
                                                       sB, hB, ηB, eB, σB, θB, γB, μB,               #browsers
                                                       sG, hG, ηG, eG, σG, θG, γG, μG,               #grazers
                                                       sS, hS, ηS, eS, σS, θS, γS, μS),              #scrapers   
                                                dense_output=False)
                
                N = sol.y
                
                C_array = N[0,:]
                M_array = N[1,:]
                T_array = N[2,:]
                
                B_array = N[3,:]
                G_array = N[4,:]
                S_array = N[5,:]

                
                #Ensure Run Reached Equilibrium-----------------------------------------------------
                thr_eq = 1e-02

                last5 = max(2, int(0.05 * N.shape[1]))
                dX = np.diff(N[:, -last5:], axis=1)
                equil = np.all(np.abs(dX) < thr_eq)

                if not equil:
                    failed = []
                    if not np.all(np.abs(np.diff(N[0, -last5:])) < thr_eq): failed.append("C")
                    if not np.all(np.abs(np.diff(N[1, -last5:])) < thr_eq): failed.append("M")
                    if not np.all(np.abs(np.diff(N[2, -last5:])) < thr_eq): failed.append("T")
                    if not np.all(np.abs(np.diff(N[3, -last5:])) < thr_eq): failed.append("B")
                    if not np.all(np.abs(np.diff(N[4, -last5:])) < thr_eq): failed.append("G")
                    if not np.all(np.abs(np.diff(N[5, -last5:])) < thr_eq): failed.append("S")

                
                #Categorize State at Equilibrium-----------------------------------------------------
                thr_st = 0.1

                Cp = C_array[-1]
                Mp = M_array[-1]
                Tp = T_array[-1]
                Fp = max(0.0, 1 - (Cp + Mp + Tp))
 
                #Benthos
                C_t = Cp >= thr_st
                M_t = Mp >= thr_st
                T_t = Tp >= thr_st

                if C_t and not M_t and not T_t:
                    ben_col = "allC"
                
                elif M_t and not C_t and not T_t:
                    ben_col = "allM"
                
                elif T_t and not C_t and not M_t:
                    ben_col = "allT"
                
                elif not C_t and not M_t and not T_t:
                    ben_col = "allF"
                
                else:
                    dom_val = max(Cp, Mp, Tp, Fp)
                
                    if dom_val == Cp:
                        ben_col = "mostlyC" if T_t else "mostlyC_noT"
                
                    elif dom_val == Mp:
                        ben_col = "mostlyM" if T_t else "mostlyM_noT"
                
                    elif dom_val == Tp:
                        ben_col = "mostlyT"
                    
                    elif dom_val == Fp:
                        ben_col = "mostlyF"
                        
                    else:
                        ben_col = "other"
                
                #Herbivores
                Bp = B_array[-1]
                Gp = G_array[-1]
                Sp = S_array[-1]

                B_t = Bp >= thr_st
                G_t = Gp >= thr_st
                S_t = Sp >= thr_st

                if B_t and not G_t and not S_t:
                    herb_col = "allB"
                
                elif G_t and not B_t and not S_t:
                    herb_col = "allG"
                
                elif S_t and not B_t and not G_t:
                    herb_col = "allS"
                
                else:
                    dom_val = max(Bp, Gp, Sp)
                
                    if dom_val == Bp:
                        herb_col = "mostlyB"
                
                    elif dom_val == Gp:
                        herb_col = "mostlyG"
                
                    elif dom_val == Sp:
                        herb_col = "mostlyS"

                    else:
                        herb_col = "other"

                
                #Save Results-----------------------------------------------------
                benthos  = (rC, αCT, μC, rM, αMT, αMC, rT)
                herbivores = (sB, hB, ηB, eB, σB, θB, γB, μB,
                              sG, hG, ηG, eG, σG, θG, γG, μG,
                              sS, hS, ηS, eS, σS, θS, γS, μS)
                state_eq = (C0, M0, T0, B0, G0, S0, 
                            Cp, Mp, Tp, Bp, Gp, Sp)
                labels   = (equil, ben_col, herb_col)
                arrays = (C_array, M_array, T_array, B_array, G_array, S_array)
#                dydt_arrays = (dC, dM, dT, dB, dG, dS)
      
                                    
                outputs.append(
                    benthos
                    + herbivores
                    + state_eq
                    + labels
                    + arrays
#                    + dydt_arrays
                 )
    
                
    benthos_c = ('rC', 'αCT', 'μC', 'rM', 'αMT', 'αMC', 'rT')
    herbivores_c = ('sB', 'hB', 'ηB', 'eB', 'σB', 'θB', 'γB', 'μB',
                    'sG', 'hG', 'ηG', 'eG', 'σG', 'θG', 'γG', 'μG',
                    'sS', 'hS', 'ηS', 'eS', 'σS', 'θS', 'γS', 'μS')
    state_eq_c = ('C0', 'M0', 'T0', 'B0', 'G0', 'S0', 
                  'Cp', 'Mp', 'Tp', 'Bp', 'Gp', 'Sp')
    labels_c = ('equil', 'ben_col', 'herb_col')
    arrays_c = ('C_array', 'M_array', 'T_array', 'B_array', 'G_array', 'S_array')
#    dydt_arrays_c = ('dC', 'dM', 'dT', 'dB', 'dG', 'dS')


    columns = (
        benthos_c
        + herbivores_c
        + state_eq_c
        + labels_c
        + arrays_c
#        + dydt_arrays_c

    )

        
    df_prime = pd.DataFrame(outputs, columns=columns)
          
    return(df_prime)
